# Multimodal Document Summarizer and RAG Application

This notebook builds a complete, end to end Multimodal Retrieval Augmented Generation (RAG) pipeline that runs entirely on a local machine inside VS Code (or any local Jupyter environment).

The pipeline follows a straight, linear flow from start to finish:

1. Read PDF files and split them into text, tables and images using the `unstructured` library.
2. Summarize the extracted tables and text with the Groq language model, since Groq is fast and well suited for large numbers of text based requests.
3. Summarize the extracted images with Google Gemini, since Gemini is a multimodal model that can understand pictures, charts and diagrams.
4. Turn every summary into a vector embedding with a free, local Hugging Face embedding model and store those embeddings in a local Chroma vector database using a Multi Vector Retriever.
5. When a user asks a question, retrieve the most relevant summaries, pull back their original full content, and ask Groq to write the final answer using that retrieved content.

Every code cell in this notebook has a markdown explanation directly above it describing what the cell does and why, plus inline comments inside the code itself, so the whole project can be followed step by step even by someone new to Retrieval Augmented Generation.


## Step 1: Install the required system tools

The Python package `unstructured` depends on two pieces of software that are **not** Python packages: `poppler` (used to read PDF pages) and `tesseract` (used for OCR, meaning it reads text that only exists as an image inside the PDF).

Because these are system tools rather than Python packages, install them once using your operating system's own package manager, not with `pip`. Open a terminal and run the command that matches your operating system.

### Windows
Install both tools with the Chocolatey package manager:
```
choco install poppler
choco install tesseract
```
If Chocolatey is not available, download the Poppler binaries and the Tesseract installer directly and add both to your system `PATH`.

### macOS
Install both tools with Homebrew:
```
brew install poppler
brew install tesseract
```

### Linux (Debian or Ubuntu based)
```
sudo apt-get update
sudo apt-get install -y poppler-utils
sudo apt-get install -y libleptonica-dev tesseract-ocr libtesseract-dev python3-pil tesseract-ocr-eng tesseract-ocr-script-latn
```

Once these system tools are installed and available on your `PATH`, restart VS Code so the new `PATH` is picked up, then continue with the Python package installation below.


## Step 2: Install the required Python packages

The cell below installs every Python package this notebook needs, grouped by what they are used for:

* `unstructured[all-docs]`, `pillow`, `pydantic`, `lxml`, `matplotlib`, `unstructured-pytesseract`: read and split the PDF into text, tables and images.
* `langchain`, `langchain-core`, `langchain-community`: build the summarization chains and the retriever.
* `langchain-groq`: connect LangChain to the Groq language model, used to summarize text and tables and to write the final answer.
* `google-genai`: connect to Google Gemini, used to summarize images.
* `sentence-transformers`: run the free, local Hugging Face embedding model used to turn summaries into vectors.
* `chromadb`: store those vectors locally in a Chroma vector database.
* `python-dotenv`: load API keys from a local `.env` file instead of hard coding them in the notebook.

`%pip` is used instead of `!pip` because `%pip` installs into the Python environment the current Jupyter kernel is actually using, which avoids accidentally installing into the wrong environment when several versions of Python are present on the machine. You only need to run this cell once per environment.


In [ ]:
%pip install "unstructured[all-docs]" pillow pydantic lxml matplotlib
%pip install unstructured-pytesseract
%pip install langchain langchain-core langchain-community
%pip install langchain-groq
%pip install google-genai
%pip install sentence-transformers
%pip install chromadb
%pip install python-dotenv


## Step 3: Import the standard library and PDF handling packages

These are the everyday Python modules plus the `unstructured` PDF parser and the image library that the rest of the notebook relies on.


In [ ]:
# Standard library
import os
import io
import re
import time
import uuid
import base64

# Third party: PDF partitioning, splits a PDF into text, table and image elements
from unstructured.partition.pdf import partition_pdf

# Third party: image handling
from PIL import Image
from IPython.display import HTML, Image as IPImage, display

# Third party: environment variable loading from a local .env file
from dotenv import load_dotenv

print("Standard library and PDF handling packages imported successfully.")


### Import the AI and retrieval building blocks

This cell imports the LangChain pieces used to build the summarization chains and the retriever, the Groq chat model, and the Google Gemini client used for image understanding.


In [ ]:
# LangChain building blocks used to build prompts and chains
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# LangChain retriever that links short summaries to their original full content
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore

# Local vector database
from langchain_community.vectorstores import Chroma

# Free, local embedding model from Hugging Face
from langchain_community.embeddings import HuggingFaceEmbeddings

# Groq chat model, used for text and table summarization and for the final answer
from langchain_groq import ChatGroq

# Google Gemini client, used to understand and summarize images
from google import genai
from google.genai import types

print("AI and retrieval building blocks imported successfully.")


## Step 4: Configure your API keys

This project uses two different AI providers:

* **Groq**, for summarizing text and tables and for writing the final answer.
* **Google Gemini**, for understanding and summarizing images.

Both keys are kept out of the notebook itself and loaded from a local `.env` file, which should never be committed to version control.

Before running the next cell:

1. In the same folder as this notebook, create a plain text file named `.env`.
2. Add two lines to that file:
   ```
   GOOGLE_API_KEY=your_google_api_key_here
   GROQ_API_KEY=your_groq_api_key_here
   ```
3. Save the file, then add `.env` to your `.gitignore` file so the keys are never pushed to a public repository.

The cell below loads that file and checks that both keys were found.


In [ ]:
# Load environment variables from the local .env file
load_dotenv()

# Read both API keys from the environment
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

# Verify that the Google Gemini key exists
if not GOOGLE_API_KEY:
    raise ValueError(
        "GOOGLE_API_KEY was not found. Make sure you created a .env file "
        "in the notebook's folder with a line like:\n"
        "GOOGLE_API_KEY=your_api_key_here"
    )

# Verify that the Groq key exists
if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY was not found. Make sure you created a .env file "
        "in the notebook's folder with a line like:\n"
        "GROQ_API_KEY=your_api_key_here"
    )

# Make sure both keys are also available as plain environment variables,
# since some libraries read them directly from the environment
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Google Gemini key loaded successfully.")
print("Groq key loaded successfully.")


### Initialize the Google Gemini client

The Gemini client is created once here and reused later whenever an image needs to be summarized.


In [ ]:
# Create a Gemini client using the API key loaded above.
# This client is reused every time an image needs to be summarized.
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

print("Google Gemini client initialized successfully.")


## Step 5: Set up local folders for the input PDFs and extracted files

This notebook works with two example PDF files. Everything lives inside a `data` folder next to this notebook, and every image or table image that gets extracted from a PDF is written to a matching folder inside `extracted_data`.

Before running the next cell, place your two PDF files here:

* `data/doc1.pdf`, the first PDF you want to analyse.
* `data/doc2.pdf`, the second PDF you want to analyse.

Feel free to rename the variables below if your files have different names.


In [ ]:
# =============================================================================
# Configure project folders
# =============================================================================

# Folder containing the source PDF documents
DATA_DIR = "data"

# Folder where extracted text, images and tables will be saved
OUTPUT_DIR = "extracted_data"

# Create the project folders if they do not already exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Paths to the PDF files
PDF_1_PATH = os.path.join(DATA_DIR, "doc1.pdf")
PDF_2_PATH = os.path.join(DATA_DIR, "doc2.pdf")

# Output folders for each PDF
EXTRACTED_DIR_1 = os.path.join(OUTPUT_DIR, "doc1")
EXTRACTED_DIR_2 = os.path.join(OUTPUT_DIR, "doc2")

# Create the extraction folders
os.makedirs(EXTRACTED_DIR_1, exist_ok=True)
os.makedirs(EXTRACTED_DIR_2, exist_ok=True)

# Display the project structure
print("=" * 60)
print("Project folders created successfully")
print("=" * 60)
print(f"Data folder          : {DATA_DIR}")
print(f"Output folder        : {OUTPUT_DIR}")
print(f"PDF 1 path            : {PDF_1_PATH}")
print(f"PDF 2 path            : {PDF_2_PATH}")
print(f"Extraction folder 1  : {EXTRACTED_DIR_1}")
print(f"Extraction folder 2  : {EXTRACTED_DIR_2}")
print("=" * 60)


## Step 6: Extract text, tables and images from the first PDF

`partition_pdf` from the `unstructured` library reads a PDF page by page and returns a list of elements. Each element knows what kind of content it represents: a title, a paragraph of narrative text, a table, an image, and so on.

Parameter meanings:

* `filename`: path to the PDF on disk.
* `strategy="hi_res"`: use the high accuracy strategy, which is slower but far better at detecting tables and images.
* `extract_images_in_pdf=True`: turn on image and table extraction.
* `extract_image_block_types`: the kinds of visual blocks to pull out, here images and tables.
* `extract_image_block_to_payload=False`: save extracted images as files on disk rather than embedding them directly inside the returned Python objects.
* `extract_image_block_output_dir`: the folder where those extracted image files are written.

This step can take a while on a large PDF because it runs full page layout analysis and OCR.


In [ ]:
raw_pdf_elements = partition_pdf(
    filename=PDF_1_PATH,                                 # path to the first PDF on your machine
    strategy="hi_res",                                    # high accuracy layout and table detection
    extract_images_in_pdf=True,                           # turn on image and table extraction
    extract_image_block_types=["Image", "Table"],          # which block types to save as image files
    extract_image_block_to_payload=False,                  # save to disk instead of embedding in memory
    extract_image_block_output_dir=EXTRACTED_DIR_1,        # local output folder for PDF 1
)

print(f"Extracted {len(raw_pdf_elements)} elements from {PDF_1_PATH}")


### Look at the raw elements from PDF 1

Printing the list shows the text content of every element together with its position in the document. This is only for inspection, so you can see what came out of the PDF before the next step sorts it into buckets.


In [ ]:
raw_pdf_elements


## Step 7: Sort the elements from PDF 1 by type

`unstructured` gives every element a Python class name that tells you what kind of content it is, for example `unstructured.documents.elements.NarrativeText` for a normal paragraph. The loop below checks the class name of every element and appends its text into the matching list, so at the end there is one clean list per content type.


In [ ]:
Header = []
Footer = []
Title = []
NarrativeText = []
Text = []
ListItem = []

for element in raw_pdf_elements:
    element_type = str(type(element))
    if "unstructured.documents.elements.Header" in element_type:
        Header.append(str(element))
    elif "unstructured.documents.elements.Footer" in element_type:
        Footer.append(str(element))
    elif "unstructured.documents.elements.Title" in element_type:
        Title.append(str(element))
    elif "unstructured.documents.elements.NarrativeText" in element_type:
        NarrativeText.append(str(element))
    elif "unstructured.documents.elements.Text" in element_type:
        Text.append(str(element))
    elif "unstructured.documents.elements.ListItem" in element_type:
        ListItem.append(str(element))

print("Headers:", len(Header))
print("Footers:", len(Footer))
print("Titles:", len(Title))
print("Narrative text blocks:", len(NarrativeText))
print("Plain text blocks:", len(Text))
print("List items:", len(ListItem))


### Pull out the image and table elements from PDF 1

The images and tables are collected separately here because they get summarized and stored differently from plain text.


In [ ]:
img = []
for element in raw_pdf_elements:
    if "unstructured.documents.elements.Image" in str(type(element)):
        img.append(str(element))

tab = []
for element in raw_pdf_elements:
    if "unstructured.documents.elements.Table" in str(type(element)):
        tab.append(str(element))

print("Images found in PDF 1:", len(img))
print("Tables found in PDF 1:", len(tab))


### View the tables found in PDF 1


In [ ]:
tab


### View the images found in PDF 1


In [ ]:
img


## Step 8: Extract text, tables and images from the second PDF

The same `partition_pdf` call is repeated for the second PDF, writing its extracted images and tables to a separate local folder so the two documents never overwrite each other's files. The Table, NarrativeText and Image elements collected from this second PDF are the ones that flow through the rest of the notebook: they get summarized, embedded and made searchable.


In [ ]:
raw_pdf_elements2 = partition_pdf(
    filename=PDF_2_PATH,                                  # path to the second PDF on your machine
    strategy="hi_res",                                     # high accuracy layout and table detection
    extract_images_in_pdf=True,                            # turn on image and table extraction
    extract_image_block_types=["Image", "Table"],           # which block types to save as image files
    extract_image_block_to_payload=False,                   # save to disk instead of embedding in memory
    extract_image_block_output_dir=EXTRACTED_DIR_2,         # local output folder for PDF 2
)

print(f"Extracted {len(raw_pdf_elements2)} elements from {PDF_2_PATH}")


### Look at the raw elements from PDF 2


In [ ]:
raw_pdf_elements2


## Step 9: Sort the elements from PDF 2 by type

For the second document, only the Table, NarrativeText and Image elements are needed, since those are the pieces that get summarized and searched later on.


### Collect the tables from PDF 2


In [ ]:
Table = []
for element in raw_pdf_elements2:
    if "unstructured.documents.elements.Table" in str(type(element)):
        Table.append(str(element))

Table


### Check how many tables were found


In [ ]:
len(Table)


### Look at one example table

Viewing a single table (the fourth one, at index 3) is a quick way to see what the raw extracted table text actually looks like before it gets summarized.


In [ ]:
Table[3]


### Collect the narrative text blocks from PDF 2


In [ ]:
Text = []
for element in raw_pdf_elements2:
    if "unstructured.documents.elements.NarrativeText" in str(type(element)):
        Text.append(str(element))

Text


### Check how many text blocks were found


In [ ]:
len(Text)


### Collect the images from PDF 2

Note: this list holds the `unstructured` text placeholders for image elements, not the actual image files. The real image files were already written to disk under `EXTRACTED_DIR_2` (and `EXTRACTED_DIR_1` for the first PDF) during the extraction step above.


In [ ]:
Image = []
for element in raw_pdf_elements2:
    if "unstructured.documents.elements.Image" in str(type(element)):
        Image.append(str(element))

Image


### Check how many images were found


In [ ]:
len(Image)


### Look at one example image element


In [ ]:
Image[0]


## Step 10: Summarize every table with Groq

Storing an entire raw table inside a vector database rarely works well for search, because tables are dense and do not read like natural language. The usual fix, used here, is to ask a language model to write a short, search friendly summary of each table. Those summaries get embedded and searched later, while the original raw table text is kept safely to one side and only shown to the user (or the final answering model) once it has actually been retrieved.

Groq is used for this step because it responds quickly and handles many summarization requests without hitting strict rate limits, which matters when a document contains dozens of tables.


### Create the table summarization prompt

This prompt gives the model clear guidelines: keep the summary concise, preserve the important numbers and relationships, focus on what will help retrieval, and never invent information that is not in the table.


In [ ]:
# =============================================================================
# Create the table summarization prompt
# =============================================================================

table_prompt = ChatPromptTemplate.from_template(
"""
You are an AI assistant for a Multimodal RAG system.

Your task is to summarize the following table.

Guidelines:

Keep the summary concise.

Preserve important entities, values, numbers, and relationships.

Focus on information useful for semantic search.

Do not invent or assume any information.

Table:

{element}
"""
)


### Initialize the Groq model and build the summarization chain

`temperature=0` keeps the summaries consistent and factual rather than creative. The chain below takes one table, inserts it into the prompt, sends it to Groq, and returns only the plain text summary.


In [ ]:
# =============================================================================
# Initialize the Groq language model
# =============================================================================

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
)

# =============================================================================
# Build the table summarization chain
# =============================================================================

table_summarize_chain = (
    {"element": lambda x: x}
    | table_prompt
    | llm
    | StrOutputParser()
)


### Summarize every extracted table

This loop runs the chain once for every table found in PDF 2 and prints a short progress message so you can follow along while it runs.


In [ ]:
# =============================================================================
# Generate summaries for all extracted tables
# =============================================================================

# List that will store the generated summaries
table_summaries = []

# Loop through every extracted table
for index, table in enumerate(Table, start=1):

    # Display the current progress
    print(f"Summarizing table {index} of {len(Table)}...")

    # Generate the summary
    summary = table_summarize_chain.invoke(table)

    # Save the summary
    table_summaries.append(summary)

print("\nTable summarization completed successfully.")


### Display the generated table summaries


In [ ]:
# =============================================================================
# Display the generated table summaries
# =============================================================================

for index, summary in enumerate(table_summaries, start=1):

    print("=" * 100)
    print(f"Table summary {index}")
    print("=" * 100)

    print(summary)
    print()


### Why use Groq instead of Gemini for table summarization?

In this project, Groq summarizes extracted tables because it provides faster inference, a more generous free usage tier, and is well suited for processing large numbers of text based requests.

Advantages of using Groq:

* **Faster response time.** Groq is optimized for extremely low latency, so table summaries are generated much faster than with most cloud hosted large language models.
* **Better suited for batch processing.** During document processing, dozens of tables may need to be summarized. Groq handles multiple summarization requests efficiently without frequently hitting strict rate limits.
* **Generous free tier.** Compared to the free Gemini API, Groq generally provides higher request limits, which is helpful when preprocessing large documents.
* **Strong text understanding.** Tables extracted from PDFs are converted into text before summarization. Since Groq specializes in text generation and reasoning, it produces concise, accurate summaries that work well for semantic retrieval.
* **Cost effective.** Groq is a good option for learning, experimentation and personal projects because it offers high performance while keeping API costs low.

Why not use Gemini for everything? Google Gemini is a multimodal model, which makes it an excellent choice for understanding images, charts and visual content. However, the free Gemini tier has stricter rate limits, which can interrupt large scale summarization tasks. For this reason, this project uses the strengths of both models: Groq summarizes the extracted text and tables, and Gemini summarizes the images. This combines the speed of Groq with the multimodal understanding of Gemini into a faster and more reliable pipeline.


## Step 11: Summarize every text block with Groq

The same idea used for tables is applied to the narrative text blocks pulled from PDF 2: each block gets a short summary that is easier to search over than the full paragraph, generated by the same Groq model used above.


### Create the text summarization prompt

This prompt instructs the model to generate concise summaries of text chunks. These summaries will later be turned into embeddings and stored in the vector database for efficient semantic retrieval.


In [ ]:
# =============================================================================
# Create the text summarization prompt
# =============================================================================

text_prompt = ChatPromptTemplate.from_template(
"""
You are an AI assistant for a Multimodal RAG system.

Your task is to summarize the following text.

Guidelines:

Keep the summary concise.

Preserve important entities, facts, dates, numbers, and relationships.

Focus on information useful for semantic search.

Do not invent or assume any information.

Text:

{element}
"""
)


### Initialize the Groq model for text summarization

A separate model instance is created here (with the same settings as the table summarizer) so this section can be understood and reused on its own.


In [ ]:
# =============================================================================
# Initialize the Groq language model
# =============================================================================

# Use Groq to generate accurate and consistent text summaries.

text_summary_model = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
)


### Build the text summarization chain

This chain performs four steps: it takes an extracted text chunk, inserts it into the prompt, sends the prompt to the Groq model, and returns only the generated summary as plain text.


In [ ]:
# =============================================================================
# Build the text summarization chain
# =============================================================================

text_summarize_chain = (
    {"element": lambda x: x}
    | text_prompt
    | text_summary_model
    | StrOutputParser()
)


### Summarize every extracted text chunk


In [ ]:
# =============================================================================
# Generate summaries for all extracted text chunks
# =============================================================================

# List that will store the generated summaries
text_summaries = []

# Loop through every extracted text chunk
for index, text in enumerate(NarrativeText, start=1):

    # Display the current progress
    print(f"Summarizing text chunk {index} of {len(NarrativeText)}...")

    # Generate the summary
    summary = text_summarize_chain.invoke(text)

    # Save the summary
    text_summaries.append(summary)

print("\nText summarization completed successfully.")


### Display the generated text summaries


In [ ]:
# =============================================================================
# Display all text summaries
# =============================================================================

for index, summary in enumerate(text_summaries, start=1):

    print("=" * 100)
    print(f"Text summary {index}")
    print("=" * 100)

    print(summary)
    print()


## Step 12: Summarize every extracted image with Google Gemini

Unlike text and tables, images cannot be directly converted into embeddings. So each extracted image is first sent to Google Gemini, a multimodal AI model capable of understanding both text and images.

Gemini analyzes the visual content and writes a concise textual description of the image. That generated summary is what gets embedded and stored in the vector database for semantic search, while the original image is preserved for later retrieval or display.

Why Google Gemini? It is a multimodal large language model that can understand images, charts, diagrams, tables, figures, screenshots and text contained within an image, which makes it a strong choice for generating meaningful image summaries in a Multimodal RAG system.

The workflow for this step is:

1. Read the extracted image from disk.
2. Convert the image into a base64 encoded string.
3. Send the image together with an instruction prompt to Gemini.
4. Generate a concise summary describing the important visual information.
5. Store the generated summary for embedding, while keeping the original image for later retrieval.


### Encode an image as base64

Vision models such as Gemini accept images as base64 text rather than raw binary files, so every image needs to be converted first.


In [ ]:
# =============================================================================
# Convert an image into a base64 string
# =============================================================================

def encode_image(image_path):
    """
    Read an image from disk and convert it into a base64 encoded string.

    Parameters
    ----------
    image_path : str
        Path to the image file.

    Returns
    -------
    str
        Base64 encoded representation of the image.
    """

    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


### Create the image summarization function

This function sends one base64 encoded image plus an instruction prompt to Gemini and returns the text summary it writes back. If the Gemini service is temporarily unavailable, the function automatically waits and retries a few times before giving up, which makes the pipeline more resilient to short outages or rate limits.


In [ ]:
# =============================================================================
# Generate an image summary using Google Gemini
# =============================================================================

def image_summarize(img_base64, prompt):
    """
    Generate a concise summary of an image using Google Gemini.

    If the Gemini service is temporarily unavailable, the function
    automatically waits and retries the request before giving up.

    Parameters
    ----------
    img_base64 : str
        Base64 encoded image.

    prompt : str
        Instruction telling Gemini how to summarize the image.

    Returns
    -------
    str
        Summary generated by Gemini.
    """

    # Maximum number of retry attempts
    MAX_RETRIES = 5

    # Wait time in seconds before retrying
    RETRY_DELAY = 10

    # Try sending the request multiple times
    for attempt in range(MAX_RETRIES):

        try:

            # Send the image and prompt to the Gemini model
            response = gemini_client.models.generate_content(
                model="gemini-flash-latest",
                contents=[
                    prompt,
                    types.Part.from_bytes(
                        data=base64.b64decode(img_base64),
                        mime_type="image/jpeg",
                    ),
                ],
            )

            # Return the generated image summary
            return response.text

        except Exception as error:

            # Display the error message
            print(f"Attempt {attempt + 1} of {MAX_RETRIES} failed.")
            print(f"Reason: {error}")

            # Retry if attempts are still available
            if attempt < MAX_RETRIES - 1:

                print(f"Waiting {RETRY_DELAY} seconds before retrying...\n")

                time.sleep(RETRY_DELAY)

            else:

                # Stop execution after all retries have failed
                raise RuntimeError(
                    "Image summarization failed after multiple retry attempts."
                )


### Create the image summarization prompt

This prompt tells Gemini exactly how to describe each extracted image. The generated summaries will later be embedded and stored in the vector database for semantic retrieval.


In [ ]:
# =============================================================================
# Create the image summarization prompt
# =============================================================================

IMAGE_SUMMARY_PROMPT = """
You are an AI assistant for a Multimodal RAG system.

Your task is to summarize the following image.

Guidelines:

Describe the important objects.

Describe any charts, diagrams, or tables.

Include important text visible in the image.

Preserve important relationships.

Keep the summary concise.

Optimize the summary for semantic retrieval.
"""


### Generate summaries for every extracted image

This function walks through every JPG image in a folder and, for each one:

1. Reads the extracted image from disk.
2. Converts it into a base64 encoded string.
3. Sends it to Google Gemini for summarization.
4. Stores both the base64 image and its generated summary.
5. Keeps going even if one particular image fails, so a single error does not stop the whole batch.


In [ ]:
# =============================================================================
# Generate summaries for all extracted images
# =============================================================================

def generate_image_summaries(image_folder):
    """
    Generate AI summaries for every extracted image in a folder.

    This function performs the following steps:

    1. Reads each extracted image from the specified folder.
    2. Converts the image into a base64 encoded string.
    3. Sends the image to Google Gemini for summarization.
    4. Stores both the base64 image and its generated summary.
    5. Continues processing even if one image fails.

    Parameters
    ----------
    image_folder : str
        Path to the folder containing the extracted JPG images.

    Returns
    -------
    tuple
        (
            image_base64_list,
            image_summaries
        )

        image_base64_list : List[str]
            Base64 encoded images.

        image_summaries : List[str]
            AI generated summaries corresponding to each image.
    """

    # List to store the base64 encoded images
    image_base64_list = []

    # List to store the generated image summaries
    image_summaries = []

    # Get all JPG images and sort them alphabetically
    image_files = sorted(
        [
            file
            for file in os.listdir(image_folder)
            if file.lower().endswith(".jpg")
        ]
    )

    print(f"Found {len(image_files)} image(s).\n")

    # Process every extracted image
    for index, image_name in enumerate(image_files, start=1):

        print(f"Processing image {index}/{len(image_files)}")
        print(f"File: {image_name}")

        # Complete path to the image
        image_path = os.path.join(image_folder, image_name)

        try:

            # Convert the image into a base64 string
            image_base64 = encode_image(image_path)

            # Save the base64 image
            image_base64_list.append(image_base64)

            # Generate an AI summary using Google Gemini
            summary = image_summarize(
                image_base64,
                IMAGE_SUMMARY_PROMPT,
            )

            # Save the generated summary
            image_summaries.append(summary)

            print("Status: success\n")

        except Exception as error:

            print("Status: failed")
            print(f"Reason: {error}\n")

            # Keep both lists aligned by storing placeholders
            image_base64_list.append(None)
            image_summaries.append("Image summarization failed.")

        # Wait briefly before processing the next image.
        # This helps avoid temporary API rate limits or server overload.
        time.sleep(2)

    print("=" * 80)
    print("Image summarization completed.")
    print(f"Total images processed: {len(image_files)}")
    print(f"Successful summaries: {len([s for s in image_summaries if s != 'Image summarization failed.'])}")
    print(f"Failed summaries: {image_summaries.count('Image summarization failed.')}")
    print("=" * 80)

    return image_base64_list, image_summaries


### Run the image summarizer

This runs the function above over every image extracted from PDF 1. Feel free to change `EXTRACTED_DIR_1` to `EXTRACTED_DIR_2` if you would rather summarize the images that came from the second PDF instead.


In [ ]:
# =============================================================================
# Generate image summaries
# =============================================================================

image_base64_list, image_summaries = generate_image_summaries(
    EXTRACTED_DIR_1
)


### Display the generated image summaries


In [ ]:
# =============================================================================
# Display the generated image summaries
# =============================================================================

for index, summary in enumerate(image_summaries, start=1):

    print("=" * 100)
    print(f"Image summary {index}")
    print("=" * 100)

    print(summary)
    print()


### View the first three images together with their summaries

Seeing the actual picture next to the text Gemini produced is the quickest way to check the summaries are accurate.


In [ ]:
# =============================================================================
# Display the first 3 images along with their generated summaries
# =============================================================================

# Get all extracted JPG images
image_files = sorted(
    [
        file
        for file in os.listdir(EXTRACTED_DIR_1)
        if file.lower().endswith(".jpg")
    ]
)

# Display only the first 3 images
for index, image_name in enumerate(image_files[:3], start=1):

    # Complete path to the image
    image_path = os.path.join(EXTRACTED_DIR_1, image_name)

    print("=" * 100)
    print(f"Image {index}")
    print(f"File name: {image_name}")
    print("=" * 100)

    # Display the extracted image
    display(IPImage(filename=image_path))

    # Display the AI generated summary
    print("Image summary:\n")
    print(image_summaries[index - 1])

    print("\n")


### Verify that every image has a matching summary

A quick sanity check confirming the two lists produced by `generate_image_summaries` stayed the same length, so every image is correctly paired with its summary.


In [ ]:
# =============================================================================
# Verify that every image has a summary
# =============================================================================

print(f"Images found: {len(image_base64_list)}")
print(f"Summaries created: {len(image_summaries)}")

if len(image_base64_list) == len(image_summaries):
    print("Every image has a corresponding summary.")
else:
    print("Some images are missing summaries.")


## Step 13: Build a Multi Vector Retriever using Chroma

At this stage, three different types of summaries have been generated from the PDFs: text summaries, table summaries and image summaries. Each summary represents its corresponding original content in a concise form that is optimized for semantic search.

Instead of embedding the large original documents directly, only these summaries are embedded. This produces better retrieval performance while reducing storage and computation. The original text, tables and images are stored separately inside an in memory document store. During retrieval, the vector database searches only the summaries, and once a relevant summary is found, the retriever returns the original content associated with that summary.

This approach is called Multi Vector Retrieval, because multiple representations (text, tables and images) are linked back to their original documents through unique document IDs.

Why use Chroma? Chroma is an open source vector database that stores embeddings locally on your machine. It requires no cloud service or API key, which makes it a good choice for learning and developing RAG applications.

Why use `BAAI/bge-base-en-v1.5`? Instead of using a paid embedding API, this notebook uses the `BAAI/bge-base-en-v1.5` embedding model from Hugging Face. It is completely free, runs locally, produces high quality embeddings, is well suited for Retrieval Augmented Generation, and needs no API key.


### Initialize the embedding model

This loads a free embedding model from Hugging Face. The model converts summaries into numerical vectors that can be stored inside the Chroma vector database. The first time this cell runs it will download the model, so it can take a moment.


In [ ]:
# =============================================================================
# Initialize the embedding model
# =============================================================================

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

print("Embedding model loaded successfully.")


### Create the Multi Vector Retriever function

This function builds a retriever that stores AI generated summaries inside the vector database and the original content inside an in memory document store. During retrieval, the summaries are searched first, and once a matching summary is found, the retriever returns the corresponding original content.


In [ ]:
# =============================================================================
# Create a Multi Vector Retriever
# =============================================================================

def create_multi_vector_retriever(
    vectorstore,
    text_summaries,
    texts,
    table_summaries,
    tables,
    image_summaries,
    images,
):
    """
    Create a Multi Vector Retriever.

    The retriever stores:

    AI generated summaries inside the vector database.
    Original content inside an in memory document store.

    During retrieval, the summaries are searched first.
    Once a matching summary is found, the retriever returns
    the corresponding original content.
    """

    # Store the original documents in memory
    store = InMemoryStore()

    # Metadata key used to associate summaries with originals
    id_key = "doc_id"

    # Create the Multi Vector Retriever
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=store,
        id_key=id_key,
    )

    # ===========================================================================
    # Helper function to add summaries and original documents
    # ===========================================================================

    def add_documents(doc_summaries, doc_contents):

        # Generate a unique ID for every original document
        doc_ids = [
            str(uuid.uuid4())
            for _ in doc_contents
        ]

        # Create LangChain Document objects containing summaries
        summary_documents = [

            Document(
                page_content=summary,
                metadata={
                    id_key: doc_ids[index]
                },
            )

            for index, summary in enumerate(doc_summaries)

        ]

        # Store summary embeddings inside Chroma
        retriever.vectorstore.add_documents(
            summary_documents
        )

        # Store the original documents in memory
        retriever.docstore.mset(
            list(zip(doc_ids, doc_contents))
        )

    # ===========================================================================
    # Add each content type
    # ===========================================================================

    if text_summaries:
        add_documents(
            text_summaries,
            texts,
        )

    if table_summaries:
        add_documents(
            table_summaries,
            tables,
        )

    if image_summaries:
        add_documents(
            image_summaries,
            images,
        )

    return retriever


### Create the Chroma vector database

This creates a local Chroma collection that will hold every summary embedding, using the Hugging Face embedding model created above.


In [ ]:
# =============================================================================
# Create the Chroma vector database
# =============================================================================

vectorstore = Chroma(
    collection_name="multimodal_rag",
    embedding_function=embedding_model,
)


### Build the Multi Vector Retriever

This puts everything together: the text summaries and their matching narrative text, the table summaries and their matching tables, and the image summaries and their matching base64 images all get added to the retriever in one call.


In [ ]:
# =============================================================================
# Build the Multi Vector Retriever
# =============================================================================

retriever_multi_vector = create_multi_vector_retriever(
    vectorstore=vectorstore,

    text_summaries=text_summaries,
    texts=NarrativeText,

    table_summaries=table_summaries,
    tables=Table,

    image_summaries=image_summaries,
    images=image_base64_list,
)

print("Multi Vector Retriever created successfully.")


## Step 14: Display images and prepare retrieved content

After creating the Multi Vector Retriever, it is useful to understand how images are stored and retrieved. Earlier in the notebook, every extracted image was converted into a base64 encoded string before being stored inside the document store. Base64 is a text representation of binary image data, which makes it easy to store and pass images between different components of the RAG pipeline.

In this section, a few small helper functions are built to:

1. Display a base64 encoded image inside the notebook.
2. Detect whether a piece of retrieved content is an image or plain text.
3. Resize images before sending them to a model.
4. Separate a list of retrieved documents into images and text so they can be processed correctly.

These helper functions are used later when building the final Multimodal RAG pipeline.


### Display a base64 encoded image


In [ ]:
# =============================================================================
# Display a base64 encoded image
# =============================================================================

def display_base64_image(img_base64):
    """
    Display a base64 encoded image inside the notebook.

    Parameters
    ----------
    img_base64 : str
        Base64 encoded image.
    """

    html = f"""
    <img
        src="data:image/jpeg;base64,{img_base64}"
        width="700"
    />
    """

    display(HTML(html))


### Test the display function on one retrieved image


In [ ]:
# =============================================================================
# Display one retrieved image
# =============================================================================

display_base64_image(image_base64_list[2])


### Display the matching image summary


In [ ]:
# =============================================================================
# Display the corresponding image summary
# =============================================================================

print(image_summaries[2])


### Check whether a string looks like base64

A quick regular expression check confirms a string only contains the characters base64 encoding uses.


In [ ]:
# =============================================================================
# Check whether a string looks like base64
# =============================================================================

def looks_like_base64(data):
    """
    Determine whether a string appears to be base64 encoded.

    Parameters
    ----------
    data : str

    Returns
    -------
    bool
    """

    return re.match(
        r"^[A-Za-z0-9+/]+={0,2}$",
        data,
    ) is not None


### Check whether base64 data represents an image

This function looks at the first few decoded bytes, known as the file signature or magic bytes, to work out whether the data is a JPEG, PNG, GIF or WEBP image.


In [ ]:
# =============================================================================
# Check whether base64 data represents an image
# =============================================================================

def is_image_data(base64_data):
    """
    Determine whether base64 data represents an image.

    The function inspects the image file signature
    (also known as the magic bytes).

    Parameters
    ----------
    base64_data : str

    Returns
    -------
    bool
    """

    image_signatures = {

        b"\xFF\xD8\xFF": "JPEG",

        b"\x89PNG\r\n\x1a\n": "PNG",

        b"GIF8": "GIF",

        b"RIFF": "WEBP",

    }

    try:

        header = base64.b64decode(base64_data)[:8]

        for signature in image_signatures:

            if header.startswith(signature):

                return True

        return False

    except Exception:

        return False


### Resize a base64 encoded image

Resizing images before sending them to a multimodal model reduces memory usage and improves inference speed.


In [ ]:
# =============================================================================
# Resize a base64 encoded image
# =============================================================================

def resize_base64_image(
    base64_string,
    size=(1300, 600),
):
    """
    Resize a base64 encoded image.

    Resizing images before sending them to the multimodal model
    reduces memory usage and improves inference speed.

    Parameters
    ----------
    base64_string : str

    size : tuple

    Returns
    -------
    str
        Resized base64 encoded image.
    """

    # Decode base64 into bytes
    image_bytes = base64.b64decode(base64_string)

    # Open the image
    image = Image.open(io.BytesIO(image_bytes))

    # Resize the image
    resized_image = image.resize(
        size,
        Image.LANCZOS,
    )

    # Save into memory
    buffer = io.BytesIO()

    resized_image.save(
        buffer,
        format=image.format,
    )

    # Convert back into base64
    return base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")


### Separate retrieved images and text

The retriever returns a mixed list of results: some entries are plain text or tables, others are base64 encoded images. This function walks through that list and splits it into two clean groups, resizing any images it finds along the way.


In [ ]:
# =============================================================================
# Separate retrieved images and text
# =============================================================================

def split_image_text_types(documents):
    """
    Separate retrieved documents into images and text.

    Images remain as base64 strings.

    Text and tables remain as plain text.

    Parameters
    ----------
    documents : list

    Returns
    -------
    dict

    {
        "images": [...],
        "texts": [...]
    }
    """

    images = []

    texts = []

    for document in documents:

        # Extract text if the object is a LangChain Document
        if isinstance(document, Document):

            document = document.page_content

        # Check whether the document is an image
        if (
            looks_like_base64(document)
            and
            is_image_data(document)
        ):

            images.append(

                resize_base64_image(document)

            )

        else:

            texts.append(document)

    return {

        "images": images,

        "texts": texts,

    }


## Step 15: Generate the final RAG answer using Groq

After the Multi Vector Retriever retrieves the most relevant documents, the returned content consists of original text, original tables and AI generated image summaries. Since every image has already been converted into a descriptive textual summary during the indexing stage, the final answer can be generated using a text only language model.

Groq is used as the final reasoning model here because it is fast, free to use, and well suited for Retrieval Augmented Generation. The overall pipeline for this step is:

1. Retrieve relevant summaries from Chroma.
2. Return the original text, tables and image summaries linked to those matches.
3. Combine everything into a single block of context.
4. Send that context together with the user's question to Groq.
5. Generate and return the final answer.


### Initialize the Groq model for the final answer


In [ ]:
# =============================================================================
# Initialize the Groq language model
# =============================================================================

# Groq will generate the final answer using the retrieved context.

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
)


### Build the Retrieval Augmented Generation prompt

This function combines the retrieved context with the user's question into a single instruction that tells Groq to answer using only what was retrieved, and to say clearly when the answer is not available in that context.


In [ ]:
# =============================================================================
# Create the Retrieval Augmented Generation prompt
# =============================================================================

def build_rag_prompt(question, retrieved_context):
    """
    Build a prompt by combining the retrieved context
    with the user's question.

    Parameters
    ----------
    question : str
        User's question.

    retrieved_context : dict
        Dictionary containing retrieved text and image summaries.

    Returns
    -------
    str
        Prompt for the Groq language model.
    """

    # Combine all retrieved text into one context
    context = "\n\n".join(retrieved_context["texts"])

    prompt = f"""
You are an expert AI assistant for a Retrieval Augmented Generation (RAG) system.

Use ONLY the retrieved context below to answer the user's question.

The retrieved context may contain:

Text
Tables
AI generated image summaries

If the answer cannot be found in the retrieved context,
clearly state that the information is unavailable.

Retrieved context:

{context}

User question:

{question}

Provide a detailed, accurate and well structured answer.
"""

    return prompt


### Generate the final response using Groq


In [ ]:
# =============================================================================
# Generate the final response using Groq
# =============================================================================

def generate_rag_response(question, retrieved_context):
    """
    Generate the final answer using the retrieved context.

    Parameters
    ----------
    question : str

    retrieved_context : dict

    Returns
    -------
    str
        Final answer generated by Groq.
    """

    # Build the prompt
    prompt = build_rag_prompt(
        question,
        retrieved_context,
    )

    # Send the prompt to Groq
    response = llm.invoke(prompt)

    return response.content


### Assemble the complete Multimodal RAG pipeline

This single function ties every earlier step together: it retrieves documents for a query, splits them into text and images, and then asks Groq to write the final answer using that retrieved context.


In [ ]:
# =============================================================================
# Build the complete Multimodal RAG pipeline
# =============================================================================

def multimodal_rag(query, retriever):
    """
    Complete Multimodal RAG pipeline.

    Steps:

    1. Retrieve relevant documents.
    2. Separate retrieved images and text.
    3. Generate the final answer using Groq.
    """

    # Retrieve relevant documents
    retrieved_documents = retriever.invoke(query)

    # Separate retrieved images and text
    retrieved_context = split_image_text_types(
        retrieved_documents
    )

    # Generate the final answer
    response = generate_rag_response(
        question=query,
        retrieved_context=retrieved_context,
    )

    return response


## Step 16: Ask questions and inspect the results

With the full pipeline assembled, it can now be asked real questions about the PDFs. Two example questions are run below, and the cells that follow show how to inspect the retrieved documents directly.


### Ask a question


In [ ]:
# =============================================================================
# Ask a question using the Multimodal RAG pipeline
# =============================================================================

query = """
Explain the retrieved figure in detail.
"""

response = multimodal_rag(
    query,
    retriever_multi_vector,
)

print(response)


### Try a second question


In [ ]:
# =============================================================================
# Ask another question
# =============================================================================

query = """
Why do we combine a pretrained retriever with a
pretrained sequence to sequence generator?
"""

response = multimodal_rag(
    query,
    retriever_multi_vector,
)

print(response)


### Display the retrieved documents for the last question

This shows how many documents the retriever actually matched for the question above, before any splitting or summarizing happens.


In [ ]:
# =============================================================================
# Display the retrieved documents
# =============================================================================

retrieved_documents = retriever_multi_vector.invoke(query)

print(f"Retrieved {len(retrieved_documents)} document(s).")


### Display the retrieved text


In [ ]:
# =============================================================================
# Display the retrieved text
# =============================================================================

retrieved_context = split_image_text_types(
    retrieved_documents
)

print("=" * 100)

print("Retrieved text\n")

print("=" * 100)

print("\n\n".join(retrieved_context["texts"]))


### Display the retrieved images


In [ ]:
# =============================================================================
# Display the retrieved images
# =============================================================================

for index, image in enumerate(
    retrieved_context["images"],
    start=1,
):

    print("=" * 100)
    print(f"Retrieved image {index}")
    print("=" * 100)

    display_base64_image(image)


## Conclusion

This project builds a complete Multimodal Retrieval Augmented Generation (Multimodal RAG) system capable of understanding and retrieving information from text, tables and images contained within PDF documents.

Instead of embedding the original PDF directly, the document is first partitioned into its individual components. Each content type is then processed independently to generate concise summaries optimized for semantic retrieval.

The extracted text and tables are summarized using Groq, while extracted images are analyzed and summarized using Google Gemini. These summaries are converted into vector embeddings using the `BAAI/bge-base-en-v1.5` embedding model and stored inside a local Chroma vector database. At the same time, the original text, tables and images are preserved inside an `InMemoryStore`, so the retriever can return the complete original content whenever it is needed.

When a user submits a query, the Multi Vector Retriever searches the embedded summaries and retrieves the corresponding original text, tables and image summaries. That retrieved context is combined into a single prompt and sent to Groq, which generates the final Retrieval Augmented Generation response.

This architecture combines the strengths of several tools:

* **Groq** generates fast summaries for text and tables and produces the final RAG answer.
* **Google Gemini** understands images and generates descriptive image summaries during the indexing stage.
* **BAAI/bge-base-en-v1.5** creates high quality embeddings locally without requiring a paid API.
* **Chroma** stores and searches vector embeddings efficiently.
* **LangChain** orchestrates the retrieval pipeline and the Multi Vector Retriever.

The result is a scalable, efficient and cost effective Multimodal RAG system capable of answering complex questions using information extracted from text, tables and images inside PDF documents.


## Tools and technologies used

* **Unstructured**: extracts text, tables, images, titles, headers and other elements from PDF documents.
* **Groq**: generates summaries for text and tables and produces the final RAG response.
* **Google Gemini**: analyzes extracted images and generates image summaries during indexing.
* **BAAI/bge-base-en-v1.5**: generates semantic embeddings for text, table and image summaries.
* **Chroma**: stores and searches vector embeddings locally.
* **LangChain**: builds the retrieval pipeline and the Multi Vector Retriever.
* **MultiVectorRetriever**: searches summary embeddings while returning the original document content.
* **InMemoryStore**: stores the original text, tables and base64 encoded images.
* **Sentence Transformers**: loads and runs the BAAI embedding model locally.
* **Pillow (PIL)**: reads, resizes and processes extracted images.
* **Python**: implements the complete Multimodal RAG pipeline.
* **Base64 encoding**: converts images into text format for storage and retrieval.
* **UUID**: generates unique identifiers linking summaries with original documents.
* **VS Code and Jupyter Notebook**: develop, run and test the project locally.


## Overall project workflow

```text
                          PDF Document
                               │
                               ▼
                Unstructured Document Parser
                               │
        ┌──────────────────────┼──────────────────────┐
        │                      │                      │
        ▼                      ▼                      ▼
      Text                  Tables                Images
        │                      │                      │
        ▼                      ▼                      ▼
   Groq Summary          Groq Summary        Gemini Summary
        │                      │                      │
        └───────────────┬──────┴──────────────┬──────┘
                        ▼
         BAAI/bge-base-en-v1.5 Embeddings
                        │
                        ▼
             Local Chroma Vector Database
                        │
                        ▼
              Multi Vector Retriever
                        │
            Search Summary Embeddings
                        │
                        ▼
 Retrieve Original Text, Tables and Image Summaries
                        │
                        ▼
          Combine Retrieved Context into Text
                        │
                        ▼
                 Groq Language Model
                        │
                        ▼
                Final Answer to the User
```
